# Fase 2 — Limpieza y Transformación
**Dataset:** Olist Brazilian E-Commerce (Kaggle)  
**Objetivo:** Aplicar transformaciones para obtener datasets limpios y consistentes, justificando cada decisión técnica.

**Estrategia:** Se creó un módulo de limpieza modular en `src/cleaning/` con un archivo por cada CSV y un archivo de utilidades comunes. La ejecución se orquesta desde `src/run_cleaning.py`.
```
src/
├── cleaning/
│   ├── utils.py                    ← funciones comunes
│   ├── clean_customers.py
│   ├── clean_geolocation.py
│   ├── clean_order_items.py
│   ├── clean_payments.py
│   ├── clean_reviews.py
│   ├── clean_orders.py
│   ├── clean_products.py
│   ├── clean_sellers.py
│   └── clean_category_translation.py
└── run_cleaning.py                 ← orquestador
```

## 1. Decisiones Globales

Estas decisiones aplican a múltiples datasets y se implementaron como funciones reutilizables en `utils.py`.

### 1.1 Normalización de ciudades
**Problema:** Los nombres de ciudades presentan inconsistencias entre tablas (geolocation, customers, sellers):
- Variantes con y sin diacríticos: "são paulo" vs "sao paulo"
- Encoding corrupto: "sa£o paulo"  
- Espacios faltantes: "sãopaulo"
- Formato incorrecto: "rio de janeiro, rio de janeiro, brasil"
- Información extra: "lages - sc"

**Decisión:** Normalizar todas las ciudades a minúsculas, sin diacríticos, sin caracteres corruptos, extrayendo solo el nombre de la ciudad.

**Justificación:** Facilita JOINs entre tablas y evita duplicados falsos. Al ser datos brasileños donde los diacríticos son inconsistentes incluso en los datos oficiales, estandarizar sin diacríticos es la opción más robusta.

### 1.2 Conversión de fechas con dayfirst=True
**Problema:** Las fechas en los CSVs raw están en formato DD/MM/YYYY (estándar brasileño). Pandas por defecto asume MM/DD/YYYY, lo que genera:
- 60% de nulos falsos (fechas con día > 12 no se pueden interpretar como mes)
- Fechas incorrectas silenciosas (ej: 02/10/2017 se interpreta como 10 de febrero en lugar de 2 de octubre)

**Decisión:** Usar `dayfirst=True` en todas las conversiones de fecha.

**Justificación:** El dataset es brasileño y usa formato DD/MM/YYYY. Sin este parámetro se pierden más de la mitad de las fechas o se interpretan incorrectamente.

## 2. Decisiones por Dataset

### 2.1 Customers
| Transformación | Justificación |
|---|---|
| Normalizar ciudades | Consistencia con las demás tablas para facilitar JOINs |
| Estados a mayúsculas | Estandarización (SP, RJ, MG) |

**Impacto:** 0 filas eliminadas. Solo transformaciones de texto.

### 2.2 Geolocation
| Transformación | Justificación |
|---|---|
| Eliminar 261,831 duplicados | Filas completamente idénticas que no aportan información |
| Eliminar coordenadas fuera de Brasil | 31 latitudes y 37 longitudes apuntando a otros países o con valores corruptos |
| Normalizar ciudades | Corregir encoding corrupto, diacríticos inconsistentes y formatos incorrectos |
| Truncar coordenadas a 6 decimales | Estandarizar precisión (~10cm), suficiente para geolocalización. Las que tienen menos se mantienen |

**Impacto:** ~262,000 filas eliminadas (26.2%), todas duplicados o datos inválidos.

### 2.3 Order Items
| Transformación | Justificación |
|---|---|
| Convertir freight_value a numérico | 1 valor no numérico (registro con freight vacío y precio máximo de R$6,735) |
| Convertir shipping_limit_date a datetime | Estaba como texto |
| Registrar 4 fechas en 2020 | Se mantienen pero se documentan como anomalía (el dataset es 2016-2018) |

**Impacto:** 0 filas eliminadas. 1 freight_value convertido a NaN.

### 2.4 Payments
| Transformación | Justificación |
|---|---|
| Eliminar 3 pagos not_defined | Son órdenes canceladas con valor 0, confirmado cruzando con tabla orders |
| Corregir 2 installments=0 en credit_card | Una tarjeta de crédito debe tener mínimo 1 cuota |

**Impacto:** 3 filas eliminadas (0.003%).

### 2.5 Reviews
| Transformación | Justificación |
|---|---|
| Convertir fechas con dayfirst=True | Formato raw DD/MM/YYYY |
| Normalizar review_creation_date a solo fecha | 99.9% tiene hora 00:00, las 85 excepciones son por horario de verano |
| Limpiar espacios en comentarios | Espacios en blanco al inicio/final de títulos y mensajes |
| Mantener review_id duplicados | Son reviews válidos del mismo cliente para diferentes órdenes del mismo pedido |

**Impacto:** 0 filas eliminadas. Solo transformaciones de formato.

### 2.6 Orders
| Transformación | Justificación |
|---|---|
| Convertir 5 columnas de fecha con dayfirst=True | Crítico: sin esto se pierden 60% de las fechas |
| Normalizar estimated_delivery_date a solo fecha | Solo registra fecha, hora siempre 00:00 |
| Normalizar order_status a minúsculas | Consistencia |

**Impacto:** 0 filas eliminadas. 0 entregas antes de compra (consistencia temporal verificada).

### 2.7 Products
| Transformación | Justificación |
|---|---|
| Corregir typo "lenght" → "length" | Error ortográfico en el dataset original |
| Etiquetar 610 productos sin categoría como "sin_categoria" | Mejor que dejar NaN para análisis y agrupaciones |
| Convertir peso=0g a NaN | Ningún producto físico pesa 0 gramos, es dato faltante |

**Impacto:** 0 filas eliminadas. 610 NaN reemplazados, 4 pesos corregidos a NaN.

### 2.8 Sellers
| Transformación | Justificación |
|---|---|
| Normalizar ciudades | Consistencia con las demás tablas. Incluye corregir "lages - sc" → "lages" |
| Estados a mayúsculas | Estandarización |

**Impacto:** 0 filas eliminadas. Solo transformaciones de texto.

### 2.9 Category Translation
| Transformación | Justificación |
|---|---|
| Corregir 4 typos en inglés | fashio→fashion, costruction→construction, confort→comfort |
| Agregar 2 categorías faltantes | pc_gamer y portateis_cozinha_e_preparadores_de_alimentos existían en products pero no en la traducción |

**Impacto:** 2 filas agregadas. 4 valores corregidos.

## 3. Ejecución de la limpieza
El siguiente código ejecuta el pipeline completo de limpieza.

In [3]:
!python ../src/run_cleaning.py

  FASE 2 — Limpieza y Transformacion
  Dataset: Olist Brazilian E-Commerce

Leyendo datos de: C:\Users\dm375\Documents\olist-data-engineering\data\raw
Guardando datos en: C:\Users\dm375\Documents\olist-data-engineering\data\clean

Cargando datasets raw...
Todos los datasets cargados

  Limpiando: Customers
   Ciudades normalizadas (minusculas, sin diacriticos)
   Estados normalizados a mayusculas

   Resultado Customers:
   Filas originales: 99,441
   Filas finales:    99,441
   Limpieza completada

  Limpiando: Geolocation
   Duplicados eliminados: 261,836
   Ciudades normalizadas (encoding, diacriticos, comas)
   Estados normalizados a mayusculas
   Coordenadas fuera de Brasil eliminadas: 33
   Coordenadas redondeadas a 6 decimales

   Resultado Geolocation:
   Filas originales: 1,000,163
   Filas finales:    738,294
   Filas eliminadas: 261,869 (26.18%)
   Limpieza completada

  Limpiando: Order Items
   freight_value convertido a numerico (1 no convertibles -> NaN)
   shipping_limi

## 4. Verificación Post-Limpieza
Cargamos los datasets limpios y verificamos que las transformaciones se aplicaron correctamente.

In [19]:
import pandas as pd
import os

CLEAN_PATH = '../data/clean/'

df_customers = pd.read_csv(f'{CLEAN_PATH}olist_customers_clean.csv')
df_geolocation = pd.read_csv(f'{CLEAN_PATH}olist_geolocation_clean.csv')
df_order_items = pd.read_csv(f'{CLEAN_PATH}olist_order_items_clean.csv')
df_payments = pd.read_csv(f'{CLEAN_PATH}olist_order_payments_clean.csv')
df_reviews = pd.read_csv(f'{CLEAN_PATH}olist_order_reviews_clean.csv')
df_orders = pd.read_csv(f'{CLEAN_PATH}olist_orders_clean.csv')
df_products = pd.read_csv(f'{CLEAN_PATH}olist_products_clean.csv')
df_sellers = pd.read_csv(f'{CLEAN_PATH}olist_sellers_clean.csv')
df_category = pd.read_csv(f'{CLEAN_PATH}product_category_name_translation_clean.csv')

print(' Datasets limpios cargados')

 Datasets limpios cargados


In [20]:
print('=== VERIFICACION: Normalización de ciudades ===\n')

for nombre, df, col in [('customers', df_customers, 'customer_city'), 
                         ('sellers', df_sellers, 'seller_city'),
                         ('geolocation', df_geolocation, 'geolocation_city')]:
    ciudades = df[col].dropna().unique()
    con_diacriticos = [c for c in ciudades if any(ch in str(c) for ch in 'áéíóúãõâêôçñü£')]
    con_comas = [c for c in ciudades if ',' in str(c)]
    con_guion = [c for c in ciudades if ' - ' in str(c)]
    
    if con_diacriticos or con_comas or con_guion:
        print(f'   {nombre}: diacriticos={len(con_diacriticos)}, comas={len(con_comas)}, guiones={len(con_guion)}')
    else:
        print(f'  {nombre}: ciudades normalizadas correctamente')

=== VERIFICACION: Normalización de ciudades ===

  customers: ciudades normalizadas correctamente
  sellers: ciudades normalizadas correctamente
  geolocation: ciudades normalizadas correctamente


In [21]:
print('=== VERIFICACION: Formato de fechas ===\n')


orders_temp = pd.read_csv(f'{CLEAN_PATH}olist_orders_clean.csv')
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
            'order_delivered_customer_date', 'order_estimated_delivery_date']:
    nulos = orders_temp[col].isna().sum()
    total = len(orders_temp)
    pct = nulos/total*100
    estado = '✅' if pct < 5 else '⚠️'
    print(f'{estado} orders.{col}: {nulos:,} nulos ({pct:.1f}%)')

print(f'\nreview_creation_date ejemplo: {df_reviews["review_creation_date"].iloc[0]}')
tiene_hora = df_reviews['review_creation_date'].str.contains(':', na=False).any()
print(f'✅ review_creation_date solo fecha: {not tiene_hora}')

=== VERIFICACION: Formato de fechas ===

✅ orders.order_purchase_timestamp: 0 nulos (0.0%)
✅ orders.order_approved_at: 160 nulos (0.2%)
✅ orders.order_delivered_carrier_date: 1,783 nulos (1.8%)
✅ orders.order_delivered_customer_date: 2,965 nulos (3.0%)
✅ orders.order_estimated_delivery_date: 0 nulos (0.0%)

review_creation_date ejemplo: 2018-01-18
✅ review_creation_date solo fecha: True


In [22]:
print('=== VERIFICACION: Payments ===\n')

not_defined = df_payments[df_payments['payment_type'] == 'not_defined']
print(f'  Pagos not_defined: {len(not_defined)} (esperado: 0)')

installments_0 = df_payments[
    (df_payments['payment_type'] == 'credit_card') & 
    (df_payments['payment_installments'] == 0)
]
print(f'  Credit cards con installments=0: {len(installments_0)} (esperado: 0)')

=== VERIFICACION: Payments ===

  Pagos not_defined: 0 (esperado: 0)
  Credit cards con installments=0: 0 (esperado: 0)


In [23]:
print('=== VERIFICACION: Products ===\n')

print(f'  Columnas: {list(df_products.columns)}')
tiene_lenght = any('lenght' in col for col in df_products.columns)
print(f'  Typo "lenght" corregido: {not tiene_lenght}')

sin_cat = df_products[df_products['product_category_name'] == 'sin_categoria']
nulos_cat = df_products['product_category_name'].isna().sum()
print(f'  Productos "sin_categoria": {len(sin_cat)} (esperado: 610)')
print(f'  Categorias nulas: {nulos_cat} (esperado: 0)')

=== VERIFICACION: Products ===

  Columnas: ['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
  Typo "lenght" corregido: True
  Productos "sin_categoria": 610 (esperado: 610)
  Categorias nulas: 0 (esperado: 0)


In [24]:
print('=== VERIFICACION: Geolocation ===\n')

dupes = df_geolocation.duplicated().sum()
print(f'  Duplicados restantes: {dupes} (esperado: 0)')

fuera_lat = df_geolocation[
    (df_geolocation['geolocation_lat'] < -33.75) | 
    (df_geolocation['geolocation_lat'] > 5.27)
]
fuera_lng = df_geolocation[
    (df_geolocation['geolocation_lng'] < -73.99) | 
    (df_geolocation['geolocation_lng'] > -34.79)
]
print(f'  Coordenadas fuera de Brasil: lat={len(fuera_lat)}, lng={len(fuera_lng)} (esperado: 0)')

max_decimales = df_geolocation['geolocation_lat'].apply(
    lambda x: len(str(x).split('.')[-1]) if '.' in str(x) else 0
).max()
print(f'  Max decimales en coordenadas: {max_decimales} (esperado: <=6)')

=== VERIFICACION: Geolocation ===

  Duplicados restantes: 0 (esperado: 0)
  Coordenadas fuera de Brasil: lat=0, lng=0 (esperado: 0)
  Max decimales en coordenadas: 6 (esperado: <=6)


In [29]:
print('=== VERIFICACION: Category Translation ===\n')

print(f'  Total categorias: {len(df_category)} (esperado: 73)')

typos_corregidos = ['fashion_female_clothing', 'construction_tools_garden', 
                    'construction_tools_tools', 'home_comfort']
for t in typos_corregidos:
    existe = t in df_category['product_category_name_english'].values
    print(f'✅ {t}: {"encontrado" if existe else "⚠️ NO encontrado"}')

nuevas = ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
for n in nuevas:
    existe = n in df_category['product_category_name'].values
    print(f'✅ {n}: {"encontrado" if existe else "⚠️ NO encontrado"}')

=== VERIFICACION: Category Translation ===

  Total categorias: 73 (esperado: 73)
✅ fashion_female_clothing: encontrado
✅ construction_tools_garden: encontrado
✅ construction_tools_tools: encontrado
✅ home_comfort: encontrado
✅ pc_gamer: encontrado
✅ portateis_cozinha_e_preparadores_de_alimentos: encontrado


In [26]:
print('=== RESUMEN COMPARATIVO RAW vs CLEAN ===\n')

RAW_PATH = '../data/raw/'
comparativo = {
    'customers': ('olist_customers_dataset.csv', 'olist_customers_clean.csv'),
    'geolocation': ('olist_geolocation_dataset.csv', 'olist_geolocation_clean.csv'),
    'order_items': ('olist_order_items_dataset.csv', 'olist_order_items_clean.csv'),
    'payments': ('olist_order_payments_dataset.csv', 'olist_order_payments_clean.csv'),
    'reviews': ('olist_order_reviews_dataset.csv', 'olist_order_reviews_clean.csv'),
    'orders': ('olist_orders_dataset.csv', 'olist_orders_clean.csv'),
    'products': ('olist_products_dataset.csv', 'olist_products_clean.csv'),
    'sellers': ('olist_sellers_dataset.csv', 'olist_sellers_clean.csv'),
    'category': ('product_category_name_translation.csv', 'product_category_name_translation_clean.csv')
}

for nombre, (raw, clean) in comparativo.items():
    df_raw = pd.read_csv(f'{RAW_PATH}{raw}', on_bad_lines='skip')
    df_clean = pd.read_csv(f'{CLEAN_PATH}{clean}')
    diff = len(df_raw) - len(df_clean)
    signo = f'-{diff:,}' if diff > 0 else f'+{abs(diff):,}' if diff < 0 else '0'
    print(f'   {nombre:15s}: {len(df_raw):>10,} → {len(df_clean):>10,} ({signo} filas)')

=== RESUMEN COMPARATIVO RAW vs CLEAN ===

   customers      :     99,441 →     99,441 (0 filas)
   geolocation    :  1,000,163 →    720,257 (-279,906 filas)
   order_items    :    112,650 →    112,650 (0 filas)
   payments       :    103,886 →    103,883 (-3 filas)
   reviews        :     99,224 →     99,224 (0 filas)
   orders         :     99,441 →     99,441 (0 filas)
   products       :     32,951 →     32,951 (0 filas)
   sellers        :      3,095 →      3,095 (0 filas)
   category       :         71 →         73 (+2 filas)


In [28]:
for nombre, df, col in [('sellers', df_sellers, 'seller_city'), 
                         ('geolocation', df_geolocation, 'geolocation_city')]:
    con_guion = df[df[col].str.contains(' - ', na=False)][col].unique()
    print(f'{nombre}: {con_guion}')

sellers: <StringArray>
[]
Length: 0, dtype: str
geolocation: <StringArray>
[]
Length: 0, dtype: str
